In [71]:
# outlier detection:
#Toy data part:
import numpy as np
import pandas as pd

from numpy.linalg import inv, norm
from sklearn.covariance import GraphicalLasso, EmpiricalCovariance
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler



# 1. Toy data generation

def make_block_precision(p=60, block_size=10, within_corr=0.35):
    """
    Create a sparse block-structured precision matrix Theta.
    """
    Theta = np.eye(p)
    n_blocks = p // block_size

    for b in range(n_blocks):
        start = b * block_size
        end = start + block_size
        for i in range(start, end):
            for j in range(start, end):
                if i != j:
                    Theta[i, j] = -within_corr / (block_size - 1)

    # Ensure positive definiteness by strengthening diagonal
    for i in range(p):
        Theta[i, i] = np.sum(np.abs(Theta[i, :])) + 0.5

    return Theta


def generate_clean_data(n=120, p=60, seed=123):
    """
    Generate clean Gaussian data from a block precision matrix.
    """
    rng = np.random.default_rng(seed)
    Theta = make_block_precision(p=p, block_size=10, within_corr=0.35)
    Sigma = inv(Theta)
    X = rng.multivariate_normal(mean=np.zeros(p), cov=Sigma, size=n)
    return X, Theta, Sigma


def inject_outliers(X, outlier_frac=0.1, shift_size=5.0, sparse_frac=0.1, seed=1234):
    """
    Inject sparse mean-shift outliers into a subset of observations.
    """
    rng = np.random.default_rng(seed)
    X_out = X.copy()
    n, p = X.shape

    n_out = int(np.floor(outlier_frac * n))
    outlier_idx = rng.choice(n, size=n_out, replace=False)

    gamma_true = np.zeros_like(X)

    for i in outlier_idx:
        n_perturb = max(1, int(np.floor(sparse_frac * p)))
        coords = rng.choice(p, size=n_perturb, replace=False)
        shifts = rng.choice([-1, 1], size=n_perturb) * shift_size
        X_out[i, coords] += shifts
        gamma_true[i, coords] = shifts

    y_true = np.zeros(n, dtype=int)
    y_true[outlier_idx] = 1
    return X_out, y_true, gamma_true, outlier_idx



# 2. Proposed toy method: alternating structure + contamination

def soft_threshold_vec(v, lam):
    """
    Elementwise soft-thresholding.
    """
    return np.sign(v) * np.maximum(np.abs(v) - lam, 0.0)


def fit_structure_aware_outlier(
    X,
    alpha_glasso=0.05,
    lambda_gamma=0.15,
    max_iter=20,
    tol=1e-4,
    standardize=True
):
    """
    A toy prototype of the proposed method.

    Steps:
    1. Initialize gamma_i = 0
    2. Fit graphical lasso on adjusted data X - gamma
    3. Update gamma_i by shrinking the residual-like quantity
    4. Iterate until convergence

    """
    X_work = X.copy()

    if standardize:
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X_work)
    else:
        Xs = X_work.copy()

    n, p = Xs.shape
    mu = Xs.mean(axis=0)
    Gamma = np.zeros((n, p))

    Theta_old = np.eye(p)

    for it in range(max_iter):
        # Step 1: adjusted data
        X_adj = Xs - Gamma

        # Step 2: estimate sparse precision matrix
        glasso = GraphicalLasso(alpha=alpha_glasso, max_iter=200)
        glasso.fit(X_adj)
        Theta = glasso.precision_
        
        mu = (Xs - Gamma).mean(axis=0)
        
        # Step 3: update contamination variables Gamma row-by-row
        Gamma_new = np.zeros_like(Gamma)
        for i in range(n):
            r_i = Xs[i] - mu
            # A simple structure-aware correction direction
            # This is a heuristic proxy for the penalized update
            correction = r_i @ Theta
            Gamma_new[i] = soft_threshold_vec(r_i, lambda_gamma)
            
        # convergence check
        diff_theta = norm(Theta - Theta_old, ord='fro')
        diff_gamma = norm(Gamma_new - Gamma, ord='fro')

        Gamma = Gamma_new
        Theta_old = Theta

        if diff_theta < tol and diff_gamma < tol:
            break

    scores = np.linalg.norm(Gamma, axis=1)

    return {
        "Theta_hat": Theta,
        "Gamma_hat": Gamma,
        "scores": scores,
        "iterations": it + 1
    }



# 3. Baselines

def mahalanobis_baseline(X):
    """
    Mahalanobis distance using empirical covariance.
    """
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    emp_cov = EmpiricalCovariance().fit(Xs)
    scores = emp_cov.mahalanobis(Xs)
    return scores


def isolation_forest_baseline(X, contamination=0.1, seed=2026):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    clf = IsolationForest(
        contamination=contamination,
        random_state=seed,
        n_estimators=200
    )
    clf.fit(Xs)

    # decision_function: larger is more normal
    # we flip sign so larger means more anomalous
    scores = -clf.decision_function(Xs)
    return scores


def lof_baseline(X, contamination=0.1, n_neighbors=20):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    clf = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        novelty=False
    )
    y_pred = clf.fit_predict(Xs)

    # negative_outlier_factor_: smaller means more anomalous
    scores = -clf.negative_outlier_factor_
    return scores



# 4. Convert scores to labels

def threshold_top_k(scores, k):
    """
    Label top-k largest scores as outliers.
    """
    n = len(scores)
    idx = np.argsort(scores)[::-1]
    y_pred = np.zeros(n, dtype=int)
    y_pred[idx[:k]] = 1
    return y_pred



def evaluate_scores(y_true, scores):
    k = int(y_true.sum())
    y_pred = threshold_top_k(scores, k)

    auc = roc_auc_score(y_true, scores)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    return {
        "AUC": auc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall
    }
    
# # =========================================================
# # 5. Single experiment(Without different lambda and gamma)
# # =========================================================
# def run_one_experiment(
#     n=120,
#     p=80,
#     outlier_frac=0.1,
#     shift_size=5.0,
#     sparse_frac=0.1,
#     seed_data=123,
#     seed_outlier=1234
# ):
#     X_clean, Theta_true, Sigma_true = generate_clean_data(n=n, p=p, seed=seed_data)
#     X, y_true, gamma_true, outlier_idx = inject_outliers(
#         X_clean,
#         outlier_frac=outlier_frac,
#         shift_size=shift_size,
#         sparse_frac=sparse_frac,
#         seed=seed_outlier
#     )

#     # Proposed method
#     fit_prop = fit_structure_aware_outlier(
#         X,
#         alpha_glasso=0.05,
#         lambda_gamma=0.18,
#         max_iter=25,
#         tol=1e-4,
#         standardize=True
#     )
#     scores_prop = fit_prop["scores"]

#     # Baselines
#     scores_maha = mahalanobis_baseline(X)
#     scores_if = isolation_forest_baseline(X, contamination=outlier_frac, seed=2026)
#     scores_lof = lof_baseline(X, contamination=outlier_frac, n_neighbors=20)

#     results = pd.DataFrame([
#         {"Method": "Proposed", **evaluate_scores(y_true, scores_prop)},
#         {"Method": "Mahalanobis", **evaluate_scores(y_true, scores_maha)},
#         {"Method": "IsolationForest", **evaluate_scores(y_true, scores_if)},
#         {"Method": "LOF", **evaluate_scores(y_true, scores_lof)},
#     ])

#     return {
#         "results": results,
#         "X": X,
#         "y_true": y_true,
#         "scores_prop": scores_prop,
#         "scores_maha": scores_maha,
#         "scores_if": scores_if,
#         "scores_lof": scores_lof,
#         "Gamma_hat": fit_prop["Gamma_hat"],
#         "Theta_hat": fit_prop["Theta_hat"],
#         "outlier_idx": outlier_idx
#     }


    
# 5. Single experiment(With different parameters gamma and lambda):

def run_one_experiment(
    n=120,
    p=80,
    outlier_frac=0.1,
    shift_size=5.0,
    sparse_frac=0.1,
    seed_data=123,
    seed_outlier=1234,
    alpha_glasso=0.05,
    lambda_gamma=0.18
):
    X_clean, Theta_true, Sigma_true = generate_clean_data(n=n, p=p, seed=seed_data)
    X, y_true, gamma_true, outlier_idx = inject_outliers(
        X_clean,
        outlier_frac=outlier_frac,
        shift_size=shift_size,
        sparse_frac=sparse_frac,
        seed=seed_outlier
    )

    # Proposed method
    fit_prop = fit_structure_aware_outlier(
        X,
        alpha_glasso=alpha_glasso,
        lambda_gamma=lambda_gamma,
        max_iter=25,
        tol=1e-4,
        standardize=True
    )
    scores_prop = fit_prop["scores"]

    # Baselines
    scores_maha = mahalanobis_baseline(X)
    scores_if = isolation_forest_baseline(X, contamination=outlier_frac, seed=2026)
    scores_lof = lof_baseline(X, contamination=outlier_frac, n_neighbors=20)

    results = pd.DataFrame([
        {"Method": "Proposed", **evaluate_scores(y_true, scores_prop)},
        {"Method": "Mahalanobis", **evaluate_scores(y_true, scores_maha)},
        {"Method": "IsolationForest", **evaluate_scores(y_true, scores_if)},
        {"Method": "LOF", **evaluate_scores(y_true, scores_lof)},
    ])

    return {
        "results": results,
        "X": X,
        "y_true": y_true,
        "scores_prop": scores_prop,
        "scores_maha": scores_maha,
        "scores_if": scores_if,
        "scores_lof": scores_lof,
        "Gamma_hat": fit_prop["Gamma_hat"],
        "Theta_hat": fit_prop["Theta_hat"],
        "outlier_idx": outlier_idx
    } 



def run_multiple_experiments(
    n_runs=20,
    mode="synthetic",   # "synthetic" or "real"
    n=120,
    p=60,
    outlier_frac=0.1,
    shift_size=5.0,
    sparse_frac=0.1,
    alpha_glasso=0.05,
    lambda_gamma=0.18,
    X=None,
    y=None,
    contamination=None
):
    all_results = []

    for r in range(n_runs):

        if mode == "synthetic":
            out = run_one_experiment(
                n=n,
                p=p,
                outlier_frac=outlier_frac,
                shift_size=shift_size,
                sparse_frac=sparse_frac,
                seed_data=100 + r,
                seed_outlier=500 + r,
                alpha_glasso=alpha_glasso,
                lambda_gamma=lambda_gamma
            )
            df = out["results"].copy()

        elif mode == "real":
            if X is None or y is None or contamination is None:
                raise ValueError("For mode='real', X, y, and contamination must be provided.")

            # Proposed
            fit_prop = fit_structure_aware_outlier(
                X,
                alpha_glasso=alpha_glasso,
                lambda_gamma=lambda_gamma,
                max_iter=25,
                tol=1e-4,
                standardize=False
            )
            scores_prop = fit_prop["scores"]

            # Baselines
            scores_maha = mahalanobis_baseline(X)
            scores_if = isolation_forest_baseline(
                X,
                contamination=contamination,
                seed=2026 + r
            )
            scores_lof = lof_baseline(
                X,
                contamination=contamination,
                n_neighbors=20
            )

            df = pd.DataFrame([
                {"Method": "Proposed", **evaluate_scores(y, scores_prop)},
                {"Method": "Mahalanobis", **evaluate_scores(y, scores_maha)},
                {"Method": "IsolationForest", **evaluate_scores(y, scores_if)},
                {"Method": "LOF", **evaluate_scores(y, scores_lof)},
            ])

        else:
            raise ValueError("mode must be either 'synthetic' or 'real'")

        df["Run"] = r + 1
        all_results.append(df)

    all_results = pd.concat(all_results, ignore_index=True)

    summary = (
        all_results.groupby("Method")[["AUC", "F1"]]
        .agg(["mean", "std"])
        .round(4)
    )

    return all_results, summary
    


In [ ]:
#20runs of the synthetic data:
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

all_results, summary = run_multiple_experiments(
    n_runs=20,
    n=80,
    p=100,
    outlier_frac=0.10,
    shift_size=1.5,
    sparse_frac=0.03
)

print(summary)

In [ ]:
#Add different parameters alpha and lambda:
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

alpha_grid = [0.01, 0.05, 0.1, 0.2, 0.5, 1]
lambda_grid = [0.08, 0.10, 0.18]

grid_results = []

for alpha in alpha_grid:
    for lam in lambda_grid:
        print(f"Running alpha={alpha}, lambda={lam}")

        try:
            _, summary = run_multiple_experiments(
                n_runs=20,
                n=80,
                p=100,
                outlier_frac=0.10,
                shift_size=1.5,
                sparse_frac=0.03,
                alpha_glasso=alpha,
                lambda_gamma=lam
            )

            prop_auc_mean = summary.loc["Proposed", ("AUC", "mean")]
            prop_auc_std = summary.loc["Proposed", ("AUC", "std")]
            prop_f1_mean = summary.loc["Proposed", ("F1", "mean")]
            prop_f1_std = summary.loc["Proposed", ("F1", "std")]

            grid_results.append({
                "alpha_glasso": alpha,
                "lambda_gamma": lam,
                "AUC_mean": prop_auc_mean,
                "AUC_std": prop_auc_std,
                "F1_mean": prop_f1_mean,
                "F1_std": prop_f1_std,
                "status": "ok"
            })

        except Exception as e:
            grid_results.append({
                "alpha_glasso": alpha,
                "lambda_gamma": lam,
                "AUC_mean": np.nan,
                "AUC_std": np.nan,
                "F1_mean": np.nan,
                "F1_std": np.nan,
                "status": f"error: {type(e).__name__}"
            })

grid_df = pd.DataFrame(grid_results)
print(grid_df)

In [ ]:
#PLot the heatmap:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#reshape to matrix
heatmap_data = grid_df.pivot(
    index="alpha_glasso",
    columns="lambda_gamma",
    values="F1_mean"
)

#sort axes
heatmap_data = heatmap_data.sort_index().sort_index(axis=1)

#plot
plt.figure(figsize=(6, 5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="Reds",
    cbar_kws={'label': 'F1 score'}
)

plt.title("Sensitivity Analysis: F1 Score")
plt.xlabel("lambda_gamma")
plt.ylabel("alpha_glasso")

plt.tight_layout()
#save the pdf plot:
plt.savefig("heatmap.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Real data part:
#Real data 1: Breast cancer  50 runs
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data
y = data.target
y_outlier = (y == 0).astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

contamination = y_outlier.mean()

all_results_bc, summary_bc = run_multiple_experiments(
    n_runs=50,
    mode="real",
    X=X_scaled,
    y=y_outlier,
    contamination=contamination,
    alpha_glasso=0.05,
    lambda_gamma=0.18
)

print(summary_bc)

In [ ]:
#real data2 thyroid:
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler

data = loadmat("/Users/lzhang/Downloads/thyroid.mat")
X = data["X"]
y = data["y"].ravel().astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

contamination = y.mean()

all_results_thyroid, summary_thyroid = run_multiple_experiments(
    n_runs=50,
    mode="real",
    X=X_scaled,
    y=y,
    contamination=contamination,
    alpha_glasso=0.05,
    lambda_gamma=0.18
)

print(summary_thyroid)

In [ ]:
# real data 3: cardio
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler

data = loadmat("/Users/lzhang/Downloads/cardio.mat")
X = data["X"]
y = data["y"].ravel().astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

contamination = y.mean()

all_results_cardio, summary_cardio = run_multiple_experiments(
    n_runs=50,
    mode="real",
    X=X_scaled,
    y=y,
    contamination=contamination,
    alpha_glasso=0.05,
    lambda_gamma=0.18
)

print(summary_cardio)

In [72]:
# real data 4: musk
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler

data = loadmat("/Users/lzhang/Downloads/musk.mat")
X = data["X"]
y = data["y"].ravel().astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

contamination = y.mean()

all_results_musk, summary_musk = run_multiple_experiments(
    n_runs=50,
    mode="real",
    X=X_scaled,
    y=y,
    contamination=contamination,
    alpha_glasso=0.3,
    lambda_gamma=0.08
)

print(summary_musk)

                    AUC              F1        
                   mean     std    mean     std
Method                                         
IsolationForest  0.9999  0.0003  0.9816  0.0259
LOF              0.6368  0.0000  0.2577  0.0000
Mahalanobis      0.9971  0.0000  0.8144  0.0000
Proposed         0.9999  0.0000  0.9691  0.0000
